In [ ]:
import os
import yfinance as yf
import time
import math
import smtplib
from email.message import EmailMessage
import pandas as pd 

C = {'G': '\033[92m', 'R': '\033[91m', 'Y': '\033[93m', 'B': '\033[94m', 'C': '\033[96m', 'W': '\033[0m'}

ATR_MULTIPLIER = 2.0 
TRAILING_STOP_PCT = 0.015     
TP_MULTIPLIER = 3.0           
PARTIAL_TP_MULTIPLIER = 1.5   
COOLDOWN_MINUTES = 15         
MAX_DAILY_LOSS_PCT = 0.02 
PORTFOLIO_CAPITAL = 100000.0  
RISK_PER_TRADE_PCT = 0.01     
entry_price = 0               
position_type = 'NONE'        
last_signal = 'NONE'
highest_price = 0             
lowest_price = 999999         
current_shares = 0  
take_profit_price = 0         
breakeven_active = False      
partial_profit_taken = False  
last_exit_time = 0            
total_pnl = 0.0
daily_pnl = 0.0
current_date_str = ""         
wins, losses = 0, 0

def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    
def send_trade_notification(signal, price, pnl, shares):
    try:
        msg = EmailMessage()
        msg.set_content(f'Bot Alert: {signal}\nPrice: {price:.2f}\nShares: {shares}\nPnL: ₹{pnl:.2f}')
        msg['Subject'] = f'Trade Alert: {signal}'
        
        SENDER_EMAIL = 'smtp.quant@gmail.com'
        APP_PASSWORD = 'iytk nlmy ooeg gkpp'
        
        msg['From'] = 'smtp.quant@gmail.com'
        msg['To'] = 'smtp.quant@gmail.com'
        
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login('smtp.quant@gmail.com', 'iytk nlmy ooeg gkpp')
            smtp.send_message(msg)
    except Exception as e:
        print(f"\n{C['R']} Email Notification Failed: {e}{C['W']}")

print(f"{C['B']} Initializing Institutional Risk Engine...{C['W']}")
master_df = yf.download('RELIANCE.NS', period='7d', interval='1m', progress=False)
print(f"{C['G']} System Online: Daily Circuit Breaker Active ({MAX_DAILY_LOSS_PCT*100}% Limit){C['W']}\n")

while True:
    try:
        today_check = time.strftime('%Y-%m-%d')
        
        if today_check != current_date_str:
            current_date_str = today_check
            daily_pnl = 0.0
            print(f"\n{C['B']}New Trading Day ({current_date_str}). Daily PnL reset to ₹0.00.{C['W']}")
            
        max_loss_allowed = -(PORTFOLIO_CAPITAL * MAX_DAILY_LOSS_PCT)
        circuit_breaker_tripped = daily_pnl <= max_loss_allowed
        new_data = yf.download('RELIANCE.NS', period='1d', interval='1m', progress=False)
        if not new_data.empty:
            master_df = pd.concat([master_df, new_data.tail(1)])
            master_df = master_df[~master_df.index.duplicated(keep='last')].tail(900) 
        else:
            time.sleep(10); continue
        master_df['SMA_45'] = master_df['Close']['RELIANCE.NS'].rolling(window=45).mean()
        master_df['SMA_190'] = master_df['Close']['RELIANCE.NS'].rolling(window=190).mean()
        master_df['SMA_750'] = master_df['Close']['RELIANCE.NS'].rolling(window=750).mean()
        delta = master_df['Close']['RELIANCE.NS'].diff()
        master_df['RSI'] = 100 - (100 / (1 + (delta.clip(lower=0).rolling(14).mean() / delta.clip(upper=0).abs().rolling(14).mean())))
        master_df['EMA_12'] = master_df['Close']['RELIANCE.NS'].ewm(span=12, adjust=False).mean()
        master_df['EMA_26'] = master_df['Close']['RELIANCE.NS'].ewm(span=26, adjust=False).mean()
        master_df['MACD'] = master_df['EMA_12'] - master_df['EMA_26']
        master_df['MACD_Signal'] = master_df['MACD'].ewm(span=9, adjust=False).mean()
        high, low, pc = master_df['High']['RELIANCE.NS'], master_df['Low']['RELIANCE.NS'], master_df['Close']['RELIANCE.NS'].shift(1)
        tr = pd.concat([high-low, (high-pc).abs(), (low-pc).abs()], axis=1).max(axis=1)
        master_df['ATR'] = tr.rolling(window=14).mean()
        latest = master_df.iloc[-1]
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        sma_45, sma_190 = latest['SMA_45'].item(), latest['SMA_190'].item()
        macro_baseline = latest['SMA_750'].item() 
        current_macd, current_macd_signal = latest['MACD'].item(), latest['MACD_Signal'].item()
        current_atr = latest['ATR'].item()
        if position_type in ['LONG', 'SHORT'] and current_shares > 0:
            stop_distance = current_atr * ATR_MULTIPLIER
            if position_type == 'LONG':
                dynamic_stop_loss = entry_price - stop_distance
                if current_price >= (entry_price + current_atr) and not breakeven_active:
                    breakeven_active = True
                    print(f"\n{C['C']} LONG BREAKEVEN: Stop-Loss moved to entry (₹{entry_price:.2f}){C['W']}")
                if breakeven_active: dynamic_stop_loss = entry_price       
            elif position_type == 'SHORT':
                dynamic_stop_loss = entry_price + stop_distance
                if current_price <= (entry_price - current_atr) and not breakeven_active:
                    breakeven_active = True
                    print(f"\n{C['C']} SHORT BREAKEVEN: Stop-Loss moved to entry (₹{entry_price:.2f}){C['W']}")
                if breakeven_active: dynamic_stop_loss = entry_price
            if not partial_profit_taken:
                trigger_partial = False
                if position_type == 'LONG' and current_price >= (entry_price + (current_atr * PARTIAL_TP_MULTIPLIER)): trigger_partial = True
                elif position_type == 'SHORT' and current_price <= (entry_price - (current_atr * PARTIAL_TP_MULTIPLIER)): trigger_partial = True
                if trigger_partial:
                    partial_profit_taken = True
                    shares_to_sell = math.floor(current_shares / 2)
                    if shares_to_sell > 0:
                        pnl = (current_price - entry_price) * shares_to_sell if position_type == 'LONG' else (entry_price - current_price) * shares_to_sell
                        total_pnl += pnl
                        daily_pnl += pnl 
                        PORTFOLIO_CAPITAL += pnl
                        current_shares -= shares_to_sell
                        print(f"\n{C['G']} PARTIAL TP ({position_type})! Banked: ₹{pnl:.2f}. Residual size running risk-free!{C['W']}")
                        send_trade_notification(f'PARTIAL_TP_{position_type}', current_price, pnl, shares_to_sell)
            if position_type == 'LONG':
                if current_price > highest_price: highest_price = current_price
                trailing_trigger = (current_price - highest_price) / highest_price <= -TRAILING_STOP_PCT
            elif position_type == 'SHORT':
                if current_price < lowest_price: lowest_price = current_price
                trailing_trigger = (lowest_price - current_price) / lowest_price <= -TRAILING_STOP_PCT
            trigger = None
            if position_type == 'LONG':
                if current_price >= take_profit_price: trigger = 'TAKE_PROFIT'
                elif current_price <= dynamic_stop_loss: trigger = 'BREAKEVEN_STOP' if breakeven_active else 'ATR_STOP_LOSS'
                elif trailing_trigger: trigger = 'TRAILING_STOP'
            elif position_type == 'SHORT':
                if current_price <= take_profit_price: trigger = 'TAKE_PROFIT'
                elif current_price >= dynamic_stop_loss: trigger = 'BREAKEVEN_STOP' if breakeven_active else 'ATR_STOP_LOSS'
                elif trailing_trigger: trigger = 'TRAILING_STOP'
            if trigger:
                pnl = (current_price - entry_price) * current_shares if position_type == 'LONG' else (entry_price - current_price) * current_shares
                total_pnl += pnl
                daily_pnl += pnl # Day 39 Update
                PORTFOLIO_CAPITAL += pnl 
                if pnl > 0: wins += 1 
                else: losses += 1
                color = C['G'] if pnl >= 0 else C['R']
                print(f"\n{color} {position_type} {trigger} HIT! Cleared {current_shares} shares at {current_price:.2f} | PnL: ₹{pnl:.2f}{C['W']}")
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger}_{position_type},{pnl}\n')
                send_trade_notification(f"{trigger}_{position_type}", current_price, pnl, current_shares)
                
                last_exit_time = time.time()
                position_type, entry_price, highest_price, current_shares, take_profit_price = 'NONE', 0, 0, 0, 0
                lowest_price = 999999
                breakeven_active, partial_profit_taken = False, False
                last_signal = 'NONE'
                time.sleep(60); continue
        macro_bullish = current_price > macro_baseline if not pd.isna(macro_baseline) else False
        macro_bearish = current_price < macro_baseline if not pd.isna(macro_baseline) else False
        if (sma_45 > sma_190) and (current_rsi < 60) and (current_macd > current_macd_signal) and macro_bullish:
            current_signal = 'LONG'
        elif (sma_45 < sma_190) and (current_rsi > 40) and (current_macd < current_macd_signal) and macro_bearish:
            current_signal = 'SHORT'
        else:
            current_signal = 'NONE'
        if current_signal != last_signal and current_shares == 0:
            if current_signal in ['LONG', 'SHORT']:
                minutes_since_exit = (time.time() - last_exit_time) / 60
                if circuit_breaker_tripped:
                    pass
                elif minutes_since_exit >= COOLDOWN_MINUTES:
                    entry_price = current_price
                    position_type = current_signal
                    risk_amount = PORTFOLIO_CAPITAL * RISK_PER_TRADE_PCT
                    stop_distance = current_atr * ATR_MULTIPLIER if current_atr > 0 else 0.5
                    current_shares = math.floor(risk_amount / stop_distance) 
                    if position_type == 'LONG':
                        dynamic_stop_loss = entry_price - stop_distance
                        take_profit_price = entry_price + (current_atr * TP_MULTIPLIER)
                        highest_price = entry_price
                    elif position_type == 'SHORT':
                        dynamic_stop_loss = entry_price + stop_distance
                        take_profit_price = entry_price - (current_atr * TP_MULTIPLIER)
                        lowest_price = entry_price
                    breakeven_active, partial_profit_taken = False, False
                    print(f"\n{C['G']} ENTRY ALERT: {position_type} at {entry_price:.2f}{C['W']}")
                    print(f"{C['C']}   ↳ Stop-Loss: ₹{dynamic_stop_loss:.2f} | Target: ₹{take_profit_price:.2f}{C['W']}")
                    with open('trade_log.csv', 'a') as f:
                        f.write(f'{time.ctime()},{current_price},{position_type},0\n')
                    send_trade_notification(position_type, current_price, 0, current_shares) 
                    last_signal = current_signal
            else:
                last_signal = current_signal
        if position_type == 'LONG': pnl_val = (current_price - entry_price) * current_shares
        elif position_type == 'SHORT': pnl_val = (entry_price - current_price) * current_shares
        else: pnl_val = 0   
        p_col = C['G'] if pnl_val >= 0 else C['R']
        mins_passed = (time.time() - last_exit_time) / 60
        if position_type != 'NONE':
            status_display = f"{C['C']}{position_type} Target: ₹{take_profit_price:.2f}{C['W']}"
        elif circuit_breaker_tripped:
            status_display = f"{C['R']} CIRCUIT BREAKER: Locked till tomorrow (Day PnL: ₹{daily_pnl:.2f}){C['W']}"
        elif mins_passed < COOLDOWN_MINUTES:
            status_display = f"{C['Y']} Lockout: {COOLDOWN_MINUTES - mins_passed:.1f}m left{C['W']}"
        else:
            status_display = f"{C['G']} Scanning | Daily PnL: ₹{daily_pnl:+.2f}{C['W']}"
        
        print(f"\r{C['B']}[{time.strftime('%H:%M:%S')}] {C['W']}"
              f"Prc: {current_price:.2f} | "
              f"{status_display} | "
              f"Trade: {p_col}₹{pnl_val:+.2f}{C['W']} | "
              f"Acct: ₹{PORTFOLIO_CAPITAL:.2f} | "
              f"W/L: {wins}/{losses}    ", end="")
    except Exception as e:
        print(f"\n{C['R']} CRITICAL ERROR: {e}{C['W']}"); time.sleep(15)
    time.sleep(60)
    

 Initializing Institutional Risk Engine...
 System Online: Daily Circuit Breaker Active (2.0% Limit)


New Trading Day (2026-06-11). Daily PnL reset to ₹0.00.
[13:08:18] Prc: 1269.50 |  Scanning | Daily PnL: ₹+0.00 | Trade: ₹+0.00 | Acct: ₹100000.00 | W/L: 0/0    

In [1]:
import sqlite3

connection = sqlite3.connect('Aplha Engine.db')
cursor = connection.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS price_ticks2 (
    tick_id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset_id INTEGER,
    time_stamp TEXT,
    price REAL
);
''')

cursor.execute("SELECT COUNT(*) FROM price_ticks2")
if cursor.fetchone()[0] == 0:
    sample_data = [
        (1, '10:00:01', 100.50), (1, '10:00:02', 101.00), (1, '10:00:03', 101.00),
        (1, '10:00:04', 102.50), (1, '10:00:05', 99.00),  (1, '10:00:06', 98.50),
        (1, '10:00:07', 98.50),  (1, '10:00:08', 99.00),  (1, '10:00:09', 100.00)
    ]
    cursor.executemany("INSERT INTO price_ticks2 (asset_id, time_stamp, price) VALUES (?, ?, ?)", sample_data)
    connection.commit()


cursor.execute('''
CREATE VIEW IF NOT EXISTS v_market_features AS
SELECT 
    time_stamp,
    price,
    AVG(price) OVER (PARTITION BY asset_id ORDER BY time_stamp ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as sma_3,
    LAG(price, 1) OVER (PARTITION BY asset_id ORDER BY time_stamp) as prev_price,
    (price - LAG(price, 1) OVER (PARTITION BY asset_id ORDER BY time_stamp)) as price_delta
FROM price_ticks2;
''')
connection.commit()


query = '''
WITH signal_data AS (
    -- The Prep Kitchen: Generate BUY/SELL/HOLD signals
    SELECT 
        time_stamp,
        CASE 
            WHEN price_delta > 0 THEN 'BUY '
            WHEN price_delta < 0 THEN 'SELL '
            ELSE 'HOLD '
        END AS signal
    FROM v_market_features
    WHERE price_delta IS NOT NULL
)
-- The Main Chef: Count how many times each signal occurred
SELECT 
    signal, 
    COUNT(*) as signal_count
FROM signal_data
GROUP BY signal
ORDER BY signal_count DESC;
'''

cursor.execute(query)
results = cursor.fetchall()

print("---  Complete Pipeline: Strategy Signal Summary ---")
print(f"{'Action':<10} | {'Total Count':<12}")
print("-" * 30)

for row in results:
    signal = row[0]
    count = row[1]
    print(f"{signal:<10} | {count:<12}")

connection.close()

---  Complete Pipeline: Strategy Signal Summary ---
Action     | Total Count 
------------------------------
BUY        | 8           
SELL       | 6           
HOLD       | 2           


In [2]:
import sqlite3
connection = sqlite3.connect('Aplha Engine.db')
cursor = connection.cursor()
query = '''
SELECT 
    time_stamp, 
    price, 
    sma_3,
    price_delta,
    CASE 
        -- RULE 1: Buy the Dip (Momentum is UP, but Price is CHEAP)
        WHEN price_delta > 0 AND price < sma_3 THEN 'BUY (Dip) '
        
        -- RULE 2: Sell the Peak (Momentum is DOWN, but Price is HIGH)
        WHEN price_delta < 0 AND price > sma_3 THEN 'SELL (Peak) '
        
        -- RULE 3: If neither strict rule is met, do nothing.
        ELSE 'HOLD '
    END AS smart_signal
FROM v_market_features
WHERE price_delta IS NOT NULL
LIMIT 15;
'''
cursor.execute(query)
results = cursor.fetchall()
print("---  Day 53: Smart Gatekeeper Signals ---")
print(f"{'Time':<12} | {'Price':<8} | {'SMA-3':<8} | {'Delta':<8} | {'Signal':<15}")
print("-" * 65)
for row in results:
    time_stamp = row[0]
    price = row[1]
    sma_3 = row[2]
    delta = row[3]
    signal = row[4]
    sma_str = f"₹{sma_3:.2f}" if sma_3 is not None else "N/A"
    delta_str = f"₹{delta:+.2f}"   
    print(f"{time_stamp:<12} | ₹{price:<7.2f} | {sma_str:<8} | {delta_str:<8} | {signal:<15}")
connection.close()

---  Day 53: Smart Gatekeeper Signals ---
Time         | Price    | SMA-3    | Delta    | Signal         
-----------------------------------------------------------------
15-06-2026 09:15:00 | ₹1264.40 | ₹1264.40 | ₹+0.00   | HOLD           
15-06-2026 09:16:00 | ₹1260.00 | ₹1262.93 | ₹-4.40   | HOLD           
15-06-2026 09:17:00 | ₹1275.50 | ₹1266.63 | ₹+15.50  | HOLD           
15-06-2026 09:18:00 | ₹1255.00 | ₹1263.50 | ₹-20.50  | HOLD           
15-06-2026 09:19:00 | ₹1280.25 | ₹1270.25 | ₹+25.25  | HOLD           
19-06-2026 12:40:03 | ₹1257.34 | ₹1264.20 | ₹-22.91  | HOLD           
19-06-2026 12:41:03 | ₹1259.44 | ₹1265.68 | ₹+2.10   | BUY (Dip)      
19-06-2026 12:42:03 | ₹1260.02 | ₹1258.93 | ₹+0.58   | HOLD           
19-06-2026 12:43:03 | ₹1255.84 | ₹1258.43 | ₹-4.18   | HOLD           
19-06-2026 12:44:03 | ₹1257.15 | ₹1257.67 | ₹+1.31   | BUY (Dip)      
19-06-2026 12:45:03 | ₹1259.76 | ₹1257.58 | ₹+2.61   | HOLD           
19-06-2026 12:46:03 | ₹1262.62 | ₹1259.84 | ₹+2

In [6]:
import sqlite3

connection = sqlite3.connect('Aplha Engine.db')
cursor = connection.cursor()

print("Building database indexes for maximum speed...")
cursor.execute('''
CREATE INDEX IF NOT EXISTS idx_asset_time 
ON price_ticks2 (asset_id, time_stamp);
''')
connection.commit()
print("Index 'idx_asset_time' created successfully!")

cursor.execute('''
SELECT * FROM price_ticks2 
ORDER BY asset_id, time_stamp DESC 
LIMIT 5;
''')

print("\n--- Fastest 5 Recent Ticks ---")
for row in cursor.fetchall():
    print(row)

# 4. Lock the vault
connection.close()

Building database indexes for maximum speed...
Index 'idx_asset_time' created successfully!

--- Fastest 5 Recent Ticks ---
(18, 1, '19-06-2026 12:49:03', 1259.32)
(17, 1, '19-06-2026 12:48:03', 1260.87)
(16, 1, '19-06-2026 12:47:03', 1257.21)
(15, 1, '19-06-2026 12:46:03', 1262.62)
(14, 1, '19-06-2026 12:45:03', 1259.76)
